In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install xgboost

from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

housing_data=pd.read_csv(r"C:/Users/000110888/OneDrive - CSULB/Desktop/housing_data.csv")

# encoding ocean_proximity
housing_data["ocean_proximity"]=housing_data["ocean_proximity"].map({
"<1H OCEAN": 1, "INLAND": 2, "NEAR BAY": 3, "NEAR OCEAN": 4, "ISLAND": 5})

# scaling target variable
housing_data["median_house_value"]=housing_data["median_house_value"]/100000

# ==========================================
# CREATING 80%/20% TRAINING/TESTING SETS
# ==========================================
train_raw, test_raw=train_test_split(housing_data, test_size=0.2, random_state=753388)

# ============================================================
# MIN-MAX SCALING OF ALL VARIABLES USING TRAINING SET ONLY
# ============================================================
train_mins=train_raw.min()
train_maxs=train_raw.max()

def min_max_scale(df, mins, maxs):
    scaled_df=(df-mins)/(maxs-mins)
    return scaled_df

train=min_max_scale(train_raw, train_mins, train_maxs)
test=min_max_scale(test_raw, train_mins, train_maxs)

# storing train-set target scaling values for inverse transformation
y_min=train_mins["median_house_value"]
y_max=train_maxs["median_house_value"]

def unscale_y(y_scaled, y_min, y_max):
    return y_scaled*(y_max-y_min)+y_min

train_x=train.drop(columns=["median_house_value"])
train_y=train["median_house_value"]
test_x=test.drop(columns=["median_house_value"])
test_y=test["median_house_value"]

def accuracy_within(actual, predicted, pct):
    return np.mean(np.abs(actual-predicted)<pct*actual)

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

def permutation_importance_rmse(model, x_data, y_data, feature_names, 
                                predict_func=None, random_state=42):
    rng = np.random.default_rng(random_state)

    if predict_func is None:
        baseline_pred=model.predict(x_data)
    else:
        baseline_pred=predict_func(model, x_data)

    baseline_rmse=rmse(y_data, baseline_pred)

    importance_rows=[]

    for feature in feature_names:
        temp_data=x_data.copy()
        temp_data[feature]=rng.permutation(temp_data[feature].values)

        if predict_func is None:
            perm_pred=model.predict(temp_data)
        else:
            perm_pred=predict_func(model, temp_data)

        perm_rmse=rmse(y_data, perm_pred)
        importance_rows.append({
            "feature": feature,
            "increase_rmse": perm_rmse-baseline_rmse
        })

    importance_df=pd.DataFrame(importance_rows).sort_values(
        by="increase_rmse", ascending=False
    ).reset_index(drop=True)

    return importance_df


def plot_actual_vs_predicted(actual_y, pred_y, title):
    x=np.arange(1, len(actual_y)+1)

    plt.figure(figsize=(10, 5))
    plt.plot(x, actual_y, linewidth=2, label="actual")
    plt.plot(x, pred_y, linewidth=2, label="predicted")
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="upper right")
    plt.show()


# ====================================
# FITTING RANDOM FOREST REGRESSION
# ====================================
rf_reg = RandomForestRegressor(n_estimators=60, max_features=5, max_leaf_nodes=100,
    random_state=753388, n_jobs=-1)
rf_reg.fit(train_x, train_y)

# displaying feature importance
imp_sorted=pd.DataFrame({
    "feature": train_x.columns,
    "importance": rf_reg.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

print("RF Regression - Feature Importance:")
print(imp_sorted)

# predicting for testing set
pred_y=rf_reg.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

# computing prediction accuracy within 10%, 15%, and 20%
acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"RF Accuracy within 10%: {acc10:.4f}")
print(f"RF Accuracy within 15%: {acc15:.4f}")
print(f"RF Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Random Forest Regression")


# ========================================
# FITTING GRADIENT BOOSTING REGRESSION
# ========================================
# FITTING GRADIENT BOOSTING REGRESSION
xgb_reg=XGBRegressor(max_depth=6, learning_rate=0.01, n_estimators=1000,
objective="reg:squarederror", verbosity=0, random_state=753388)
xgb_reg.fit(train_x, train_y)

# displaying feature importance
imp_sorted=pd.DataFrame({
    "Feature": train_x.columns,
    "Gain": xgb_reg.feature_importances_
}).sort_values(by="Gain", ascending=False)

print("XGB Regression - Feature Importance:")
print(imp_sorted)

# predicting for testing set
pred_y=xgb_reg.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

# computing prediction accuracy within 10%, 15%, and 20%
acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print("XGB Accuracy within 10%:", round(acc10, 4))
print("XGB Accuracy within 15%:", round(acc15, 4))
print("XGB Accuracy within 20%:", round(acc20, 4))

# plotting actual and predicted values
plot_actual_vs_predicted(actual_y, pred_y, "Gradient Boosting Regression")

# ========================================================
# FITTING SUPPORT VECTOR REGRESSION WITH LINEAR KERNEL
# ========================================================
svr_linear=SVR(kernel="linear")
svr_linear.fit(train_x, train_y)

# displaying feature importance
imp_sorted=pd.DataFrame({
    "feature": train_x.columns,
    "importance": np.abs(svr_linear.coef_.ravel())
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

print("SVR (Linear) - Feature Importance:")
print(imp_sorted)

# predicting for testing set
pred_y=svr_linear.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"SVR (Linear) Accuracy within 10%: {acc10:.4f}")
print(f"SVR (Linear) Accuracy within 15%: {acc15:.4f}")
print(f"SVR (Linear) Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Support Vector Regression (Linear Kernel)")

# ============================================================
# FITTING SUPPORT VECTOR REGRESSION WITH POLYNOMIAL KERNEL
# ============================================================
svr_poly=SVR(kernel="poly")
svr_poly.fit(train_x, train_y)

importance=permutation_importance_rmse(svr_poly, test_x, test_y, 
test_x.columns, random_state=42)

print("SVR (Polynomial) - Feature Importance:")
print(importance)

pred_y=svr_poly.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"SVR (Polynomial) Accuracy within 10%: {acc10:.4f}")
print(f"SVR (Polynomial) Accuracy within 15%: {acc15:.4f}")
print(f"SVR (Polynomial) Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Support Vector Regression (Polynomial Kernel)")

# ========================================================
# FITTING SUPPORT VECTOR REGRESSION WITH RADIAL KERNEL
# ========================================================
svr_radial=SVR(kernel="rbf")
svr_radial.fit(train_x, train_y)

importance=permutation_importance_rmse(svr_radial, test_x, test_y, 
test_x.columns, random_state=42)

print("SVR (Radial) - Feature Importance:")
print(importance)

pred_y=svr_radial.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"SVR (Radial) Accuracy within 10%: {acc10:.4f}")
print(f"SVR (Radial) Accuracy within 15%: {acc15:.4f}")
print(f"SVR (Radial) Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Support Vector Regression (Radial Kernel)")

# =========================================================
# FITTING SUPPORT VECTOR REGRESSION WITH SIGMOID KERNEL
# =========================================================
svr_sigmoid=SVR(kernel="sigmoid")
svr_sigmoid.fit(train_x, train_y)

importance=permutation_importance_rmse(svr_sigmoid, test_x, test_y, 
test_x.columns, random_state=42)

print("SVR (Sigmoid) - Feature Importance:")
print(importance)

pred_y=svr_sigmoid.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"SVR (Sigmoid) Accuracy within 10%: {acc10:.4f}")
print(f"SVR (Sigmoid) Accuracy within 15%: {acc15:.4f}")
print(f"SVR (Sigmoid) Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Support Vector Regression (Sigmoid Kernel)")

# =========================================
# FITTING K-NEAREST NEIGHBOR REGRESSION
# =========================================
knn_reg=KNeighborsRegressor()
knn_reg.fit(train_x, train_y)

importance=permutation_importance_rmse(knn_reg, test_x, test_y, 
test_x.columns, random_state=42)

print("KNN Regression - Feature Importance:")
print(importance)

pred_y=knn_reg.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"KNN Accuracy within 10%: {acc10:.4f}")
print(f"KNN Accuracy within 15%: {acc15:.4f}")
print(f"KNN Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "K-Nearest Neighbor Regression")

# ================================================
# FITTING ARTIFICIAL NEURAL NETWORK REGRESSION
# ================================================
ann_reg=MLPRegressor(hidden_layer_sizes=(3,), activation="logistic",
solver="adam", max_iter=2000, random_state=753388)
ann_reg.fit(train_x, train_y)

importance=permutation_importance_rmse(ann_reg, test_x, test_y, 
test_x.columns, random_state=42)

print("ANN Regression - Feature Importance:")
print(importance)

pred_y=ann_reg.predict(test_x)

actual_y=unscale_y(test_y, y_min, y_max)
pred_y=unscale_y(pred_y, y_min, y_max)

acc10=accuracy_within(actual_y, pred_y, 0.10)
acc15=accuracy_within(actual_y, pred_y, 0.15)
acc20=accuracy_within(actual_y, pred_y, 0.20)

print(f"ANN Accuracy within 10%: {acc10:.4f}")
print(f"ANN Accuracy within 15%: {acc15:.4f}")
print(f"ANN Accuracy within 20%: {acc20:.4f}")

plot_actual_vs_predicted(actual_y, pred_y, "Artificial Neural Network Regression")